# Faruq-v3 — fresh DLRBC seed-42 decision
Run this only after B0_FRESH, LRLIN_FRESH, and DLRBC_FRESH have completed. It performs no training and does not access test data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
BRANCH='codex/dlrbc-fresh-screening'
REPO=Path('/content/coffee-bean-detection'); os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result=subprocess.run(clone,cwd='/content')
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    time.sleep(2)
else: raise RuntimeError('Git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True,cwd='/content')
for name in list(sys.modules):
    if name=='coffee_detector' or name.startswith('coffee_detector.'): sys.modules.pop(name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.drive_project import resolve_drive_project_root
PROJECT=resolve_drive_project_root()
OUTPUT=PROJECT/'experiments/faruq-v3-dlrbc-fresh-v1'
RESULTS=[OUTPUT/'val_reports'/f'{arm}_seed42_result.json' for arm in ('B0_FRESH','LRLIN_FRESH','DLRBC_FRESH')]
missing=[str(path) for path in RESULTS if not path.is_file()]
if missing: raise FileNotFoundError(f'Arm belum lengkap: {missing}')
print('RESULTS:',*[str(path) for path in RESULTS],sep='\n- ')

In [ ]:
from coffee_detector.experiments.decide_faruq_v3_dlrbc_fresh import build_fresh_dlrbc_decision
SUMMARY=OUTPUT/'val_reports/dlrbc_fresh_seed42_decision.json'
decision=build_fresh_dlrbc_decision(RESULTS,SUMMARY)
print(json.dumps(decision,indent=2,ensure_ascii=False))
print('SUMMARY:',SUMMARY)
print('TEST OPENED:',decision['test_images_accessed'])